In [36]:
import folium
import ipywidgets as widgets
from IPython.display import display
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
import base64
import os.path as pa
import os
from scipy.stats import linregress

In [34]:

# Load your data
file_path = '/home/apidell_03/Climate_Indek/output/TREND_AllSTAT_TEMP.csv'
data = pd.read_csv(file_path)

# Define the list of indices
indices_list = ['TMm','TMx','TMn', 'TXm', 'TXx', 'TXn', 'TNx', 'TNn', 'TNm','DTR','ETR']

# Function to create the map
def create_map(data, selected_index):
    map_center  = [data['Lat'].mean(), data['Lon'].mean()]
    mean_values = data.groupby('StatName')[selected_index].mean()
    
    # Create a map centered around the average coordinates
    weather_map = folium.Map(location=map_center, zoom_start=5)

    # Normalize the mean values for coloring
    min_value = mean_values.min()
    max_value = mean_values.max()
    cmap = plt.cm.coolwarm

    # Add points for each weather station with color based on the selected index
    for _, row in data.iterrows():
        mean_value = mean_values[row['StatName']]
        color = mcolors.to_hex(cmap((mean_value - min_value) / (max_value - min_value)))
        popup_text = f"""
        <b>Station:</b> {row['StatName']}<br>
        <b>WMOId:</b> {row['WMOId']}<br>
        <b>StrDate:</b> {row['StrDate']}<br>
        <b>EndDate:</b> {row['EndDate']}<br>
        <b>{selected_index}:</b> {round(row[selected_index], 2)}<br>
        """
        folium.CircleMarker(
            location=[row['Lat'], row['Lon']],
            radius=6,
            popup=folium.Popup(popup_text, max_width=300),
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.9
        ).add_to(weather_map)

    # Save the map to an HTML file
    map_path = f'/home/apidell_03/Climate_Indek/html/weather_map_{selected_index}.html'
    weather_map.save(map_path)
    
    return map_path

# Create a dropdown menu for selecting indices
dropdown = widgets.Dropdown(
    options=indices_list,
    value='TNX',
    description='Index:',
    disabled=False,
)

# Create a button to generate the map
button = widgets.Button(description="Generate Map")

# Display the dropdown menu and button
display(dropdown, button)

# Define the button click event handler
def on_button_click(b):
    selected_index = dropdown.value
    map_path = create_map(data, selected_index)
    display(f"Download the map: [Download Map](sandbox:{map_path})")

# Attach the event handler to the button
button.on_click(on_button_click)

FileNotFoundError: [Errno 2] No such file or directory: '/home/apidell_03/Climate_Indek/output/TREND_AllSTAT_TEMP.csv'

In [37]:
# PATH LOCATION
WorkDir        = '/home/apidell_03/Climate_Indek'
DataDir        = pa.join(WorkDir,'Data')
RawDataPath    = pa.join(DataDir,'raw')
SrcDir         = pa.join(WorkDir,'src')
Fig            = pa.join(WorkDir,'fig')

# SETUP DIRECTORY FOR OUTPUT LOCATION
obs_dir         = pa.join(DataDir,"obs")
clean_dir       = pa.join(DataDir,"clean") 
outdata_dir     = pa.join(DataDir,"quality")
report_dir      = pa.join(WorkDir, "report")

# SETUP DIRECTORY FOR OUTPUT INDECES
OptIdx_dir      = pa.join(WorkDir,'output')

if pa.exists(obs_dir)     == False: os.makedirs(obs_dir)
if pa.exists(clean_dir)   == False: os.makedirs(clean_dir)
if pa.exists(outdata_dir) == False: os.makedirs(outdata_dir)
if pa.exists(report_dir)  == False: os.makedirs(report_dir)
if pa.exists(OptIdx_dir)  == False: os.makedirs(OptIdx_dir)

In [38]:
# Example usage:
# Assuming df is your DataFrame
data_path = '/home/apidell_03/Climate_Indek/output/INDEK_AllSTAT_TEMP.csv'
Data = pd.read_csv(data_path)
DataGrouped = Data.groupby(['WMOId', 'StatName', 'Lat', 'Lon'])

In [39]:
# Define a function to calculate the trendline for each group and each temperature variable
def calculate_trend_all_vars(group):
    results = {}
    temperature_vars = ['TMm', 'TMx', 'TMn', 'TXm', 'TXx', 'TXn', 'TNx', 'TNn', 'TNm','DTR','ETR']
    for var in temperature_vars:
        slope, intercept, r_value, p_value, std_err = linregress(group['YEAR'], group[var])
        results[f'{var}_slope']     = slope
        results[f'{var}_intercept'] = intercept
        results[f'{var}_r_value']   = r_value
        results[f'{var}_p_value']   = p_value
        results[f'{var}_std_err']   = std_err
    return pd.Series(results)

# Apply the function to each group
TrendLineDf = DataGrouped.apply(calculate_trend_all_vars).reset_index()


In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def sens_slope(y, x):
    slopes = []
    for i in range(len(x) - 1):
        for j in range(i + 1, len(x)):
            slopes.append((y[j] - y[i]) / (x[j] - x[i]))
    return np.median(slopes)

def calculate_sens_slope(group):
    results = {}
    temperature_vars = ['TMm', 'TMx', 'TMn', 'TXm', 'TXx', 'TXn', 'TNx', 'TNn', 'TNm','DTR','ETR']
    for var in temperature_vars:
        try:
            if var not in group.columns or 'YEAR' not in group.columns:
                print(f"Missing column {var} or YEAR in group {group.name}")
                results[f'{var}_slope'] = np.nan
                continue
            x = group['YEAR'].values
            y = group[var].values
            slope = sens_slope(y, x)
            results[f'{var}_slope'] = slope
        except Exception as e:
            print(f"Error processing {var} in group {group.name}: {e}")
            results[f'{var}_slope'] = np.nan
    return pd.Series(results)

# Load the dataset
data_path = '/home/apidell_03/Climate_Indek/output/INDEK_AllSTAT_TEMP.csv'
Data = pd.read_csv(data_path)

# Remove rows with missing values in the temperature columns
temperature_vars = ['TMm', 'TMx', 'TMn', 'TXm', 'TXx', 'TXn', 'TNx', 'TNn', 'TNm','DTR','ETR']
cleaned_data = Data.dropna(subset=temperature_vars)

# Check for missing columns before applying the function
missing_columns = [var for var in temperature_vars if var not in cleaned_data.columns]
if missing_columns:
    print(f"Missing columns in the dataset: {missing_columns}")

# Calculate Sen's slopes
sens_slopes = cleaned_data.groupby(['WMOId', 'StatName', 'Lat', 'Lon']).apply(calculate_sens_slope).reset_index()

# Save the calculated slopes to a CSV file
sens_slopes.to_csv('/home/apidell_03/Climate_Indek/output/SLOPE_AllSTAT_TEMP.csv', index=False)


In [41]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import pandas as pd
import os.path as pa
import os
import glob

# PATH LOCATION
WorkDir        = '/home/apidell_03/Climate_Indek'
DataDir        = pa.join(WorkDir,'Data')
RawDataPath    = pa.join(DataDir,'raw')
SrcDir         = pa.join(WorkDir,'src')
Fig            = pa.join(WorkDir,'fig')

# SETUP DIRECTORY FOR OUTPUT LOCATION
obs_dir         = pa.join(DataDir,"obs")
clean_dir       = pa.join(DataDir,"clean") 
outdata_dir     = pa.join(DataDir,"quality")
report_dir      = pa.join(WorkDir, "report")

# SETUP DIRECTORY FOR OUTPUT INDECES
OptIdx_dir      = pa.join(WorkDir,'output')

if pa.exists(obs_dir)     == False: os.makedirs(obs_dir)
if pa.exists(clean_dir)   == False: os.makedirs(clean_dir)
if pa.exists(outdata_dir) == False: os.makedirs(outdata_dir)
if pa.exists(report_dir)  == False: os.makedirs(report_dir)
if pa.exists(OptIdx_dir)  == False: os.makedirs(OptIdx_dir)


In [42]:
FileExist         = glob.glob(pa.join(RawDataPath,'*.csv'))
NameFileExist     = FileExist[0].split('/')[-1]
print(f'File Exist : {NameFileExist}')

#Ekstrak Informasi Data Yang ada Di Dataset
StrDate           = FileExist[0].split('/')[-1].split('_')[-2]
EndDate           = FileExist[0].split('/')[-1].split('_')[-1].split('.')[-2]
print(f'Start Date : {StrDate}, End Date : {EndDate}')

Metadata          = pd.read_csv(pa.join(DataDir,f'METADATA_{StrDate}_{EndDate}.csv'))

File Exist : FKLIM_DAILY_1981-01-01_2023-01-31.csv
Start Date : 1981-01-01, End Date : 2023-01-31


In [43]:
import os
import pandas as pd
import matplotlib.pyplot as plt

import numpy as np

# Read Indeces Dataset
data_path = os.path.join('/home/apidell_03/Climate_Indek/output', 'INDEK_AllSTAT_TEMP.csv')
Data      = pd.read_csv(data_path)
SlopeBin  = []

# List of temperature variables
temperature_vars = ['TMm', 'TMx', 'TMn', 'TXm', 'TXx', 'TXn', 'TNx', 'TNn', 'TNm', 'DTR', 'ETR']

for i in Metadata['WMO_ID']:
    filtered_data = Data[Data['WMOId'] == int(i)].dropna(subset=temperature_vars)
    MetadataSel   = Metadata[Metadata['WMO_ID'] == i]
    StatName      = MetadataSel['NAME'].values[0]
    FigName       = f'PLOT_TREN_{StatName}.png'

    IdxDir = os.path.join('/home/apidell_03/Climate_Indek/output', str(i))
    
    if not os.path.exists(IdxDir):
        os.makedirs(IdxDir)

    # Create subplots
    fig, axs = plt.subplots(3, 4, figsize=(21, 9))  # Adjusted to 3 rows and 4 columns for better layout

    SlopDf = {}
    # Loop through each temperature variable and plot
    for idx, varname in enumerate(temperature_vars):
        if varname in filtered_data.columns:
            x = filtered_data['YEAR']
            y = filtered_data[varname]

            # Convert Data to Numpy
            X = x.to_numpy().reshape(-1, 1)
            Y = y.to_numpy()

            # Model
            model = LinearRegression()
            model.fit(X, Y)

            slope = model.coef_[0]
            intercept = model.intercept_
            SlopDf[varname] = {'slope': slope, 'intercept': intercept}

            # Regression line
            Y_pred = model.predict(X)

            # Scatter plot of original data
            ax = axs[idx // 4, idx % 4]
            ax.scatter(X, Y, label=f'Data Asli {varname}', alpha=0.5)
            ax.plot(X, Y_pred, linestyle='--', linewidth=1, label=f'Regresi Linear {varname}', color='red')
            ax.annotate(f'Slope: {slope:.2f}\nIntercept: {intercept:.2f}', 
                        xy=(0.05, 0.75), xycoords='axes fraction', fontsize=10, 
                        bbox=dict(boxstyle="round,pad=0.3", edgecolor="black"))

            # Labeling
            ax.set_title(varname, fontsize=10)
            ax.grid(True)

    # Set common labels
    fig.text(0.5, 0.04, 'Year', ha='center', fontsize=12)
    fig.suptitle(f'Stasiun {StatName}', fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    # Save the figure
    FileOut = os.path.join(IdxDir, FigName)
    plt.savefig(FileOut, bbox_inches='tight')
    plt.close()

    SlopDf['StatName'] = StatName
    SlopDf['WMOId']    = i
    SlopDf['StrDate']  = MetadataSel['START_DATA'].values[0]
    SlopDf['EndDate']  = MetadataSel['END_DATA'].values[0]
    SlopDf['Lat']      = MetadataSel['CURRENT_LATITUDE'].values[0]
    SlopDf['Lon']      = MetadataSel['CURRENT_LONGITUDE'].values[0]

    SlopeBin.append(pd.DataFrame(SlopDf))  # Corrected to create DataFrame from a list of dicts

# Combine all dataframes in SlopeBin
# combined_slope_df = pd.concat(SlopeBin, ignore_index=True)

In [45]:
idx

10

In [46]:
varname

'ETR'

In [44]:
DfJoin           = pd.concat(SlopeBin)
slope_DfJoin     = DfJoin.loc['slope']
intercept_DfJoin = DfJoin.loc['intercept']
slope_DfJoin.to_csv('/home/apidell_03/Climate_Indek/output/SLOPE_AllSTAT_TEMP.csv',index=False)
intercept_DfJoin.to_csv('/home/apidell_03/Climate_Indek/output/INTERCEPT_AllSTAT_TEMP.csv',index=False)